<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/06_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 06: HYPERPARAMETER TUNING & MODEL PERSISTENCE
**Workflow Phases 09 & 11** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Formulate cross-validation using **TimeSeriesSplit (5-fold)** to avoid forward temporal leakage.
2. Optimize ensemble hyperparameters (`n_estimators`, `max_depth`, `learning_rate`, `subsample`, `min_samples_split`).
3. Mitigate variance on noisy daily admission spikes.
4. Save the final optimized model pipeline and metadata (`models/patient_flow_model.pkl`).


In [ ]:
# 1. IMPORT LIBRARIES & CROSS-VALIDATION MODULES
import pandas as pd
import numpy as np
import joblib
import sys
import os

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor

sys.path.append('src')
from train import time_based_train_test_split, evaluate_predictions
from preprocessing import get_feature_columns

print("Hyperparameter tuning modules loaded.")


In [ ]:
# 2. LOAD TRAINING DATA
df = pd.read_csv('data/daily_patient_flow.csv')
feature_cols = get_feature_columns()
target_col = 'target_tomorrow_admissions'

train_df, test_df = time_based_train_test_split(df, split_date='2024-01-01')
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]


In [ ]:
# 3. CONFIGURE TIMESERIESSPLIT CROSS-VALIDATION
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"Fold {fold+1}: Training window {len(train_idx)} days -> Validation window {len(val_idx)} days")


In [ ]:
# 4. RANDOMIZED SEARCH CROSS-VALIDATION
param_dist = {
    'n_estimators': [80, 100, 120, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.08],
    'subsample': [0.75, 0.85, 0.95],
    'min_samples_split': [2, 5, 10]
}

base_gb = GradientBoostingRegressor(random_state=42)

search = RandomizedSearchCV(
    base_gb,
    param_distributions=param_dist,
    n_iter=10,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
print(f"Optimal Hyperparameters: {search.best_params_}")
print(f"Best Cross-Validation MAE: {-search.best_score_:.3f} patients")


In [ ]:
# 5. OUT-OF-SAMPLE TEST EVALUATION
tuned_preds = np.clip(best_model.predict(X_test), 0, None)
tuned_eval = evaluate_predictions(y_test, tuned_preds)

print("=== FINAL TUNED MODEL OUT-OF-SAMPLE PERFORMANCE ===")
for k, v in tuned_eval.items():
    print(f" • {k}: {v}")


In [ ]:
# 6. PERSIST PRODUCTION MODEL ARTIFACT
bundle = joblib.load('models/patient_flow_model.pkl')
print("Model bundle verified on disk.")
print("Bundle keys:", list(bundle.keys()))
